In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display
import cv2
import numpy as np
import logging
import random

# Configure logging
logging.basicConfig(level=logging.INFO)

IMG_SUFFIXES = [".jpg", ".jpeg", ".png", ".bmp", ".tiff"]

def load_img_rgb(path: Path) -> np.ndarray:
    
    if path.suffix.lower() not in IMG_SUFFIXES:
        logging.error(f"Unsupported image format: {path.suffix}. Supported formats are: {IMG_SUFFIXES}")
        raise ValueError(f"Unsupported image format: {path.suffix}. Supported formats are: {IMG_SUFFIXES}")
    
    abs_path = path.resolve()
    if not abs_path.exists():
        logging.error(f"Image file not found: {abs_path}")
        raise FileNotFoundError(f"Image file not found: {abs_path}")
    
    img_pil = Image.open(abs_path)
    img_np = np.array(img_pil)
    
    if img_np.ndim == 2:  # Grayscale image
        img_np_rgb = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
    elif img_np.shape[2] == 4:  # RGBA image
        img_np_rgb = cv2.cvtColor(img_np, cv2.COLOR_RGBA2RGB)
    elif img_np.shape[2] == 3:  # RGB image
        img_np_rgb = img_np
    else:
        logging.error(f"Unsupported image shape: {img_np.shape}")
        raise ValueError(f"Unsupported image shape: {img_np.shape}")
    
    return img_np_rgb

def display_img(img_np_rgb: np.ndarray) -> None:
    if img_np_rgb is None or not isinstance(img_np_rgb, np.ndarray):
        logging.error("Invalid image data provided for display.")
        raise ValueError("Invalid image data provided for display.")
    
    img_pil = Image.fromarray(img_np_rgb)
    display(img_pil)






img_dir = Path("./data/sample3/")
sample_img_path = random.choice(list(img_dir.glob("*")))
sample_np_img = load_img_rgb(sample_img_path)
display_img(sample_np_img)



In [ ]:
import sys
from pathlib import Path
from typing import Optional, Tuple, List

import cv2
import numpy as np
from PIL import Image

# MobileSAM: SAM互換API
from mobile_sam import sam_model_registry, SamPredictor

# ============== 設定（グローバル変数） ==============
MOBILE_SAM_WEIGHTS = Path("./models/mobile_sam.pt")  # MobileSAMの重みパス
MODEL_TYPE = "vit_t"                          # MobileSAMはTiny-ViT系 ("vit_t")
DEVICE = "cpu"

# 入力画像パス
IMG_PATH = Path("./data/sample3/s-IMG_4154.jpg")

# プロンプト指定（いずれか片方を使用、両方Noneなら中央点を使用）
POINT_PROMPT: Optional[Tuple[int, int]] = None   # 例: (900, 1200)
BOX_PROMPT: Optional[Tuple[int, int, int, int]] = None  # 例: (200, 150, 1600, 2200)

# マスク→ポリゴン抽出のパラメータ
MIN_AREA = 10_000           # 小さすぎる領域を除外（px）
EPSILON_RATIO = 0.01        # approxPolyDP の近似率
USE_MIN_AREA_RECT = True    # 四隅を4点で安定化

# ============== ユーティリティ ==============
def load_mobilesam(weights_path: Path, model_type: str = MODEL_TYPE, device: str = DEVICE):
    assert weights_path.exists(), f"Weights not found: {weights_path}"
    sam = sam_model_registry[model_type](checkpoint=str(weights_path))
    sam.to(device=device)
    return SamPredictor(sam)

def pil_to_ndarray(img: Image.Image) -> np.ndarray:
    return cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)

def ndarray_to_pil(img_bgr: np.ndarray) -> Image.Image:
    return Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

def predict_mask(
    predictor: SamPredictor,
    img_bgr: np.ndarray,
    point_prompt: Optional[Tuple[int, int]] = None,
    box_prompt: Optional[Tuple[int, int, int, int]] = None,
    multimask_output: bool = False,
) -> np.ndarray:
    predictor.set_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

    if point_prompt is None and box_prompt is None:
        h, w = img_bgr.shape[:2]
        point_prompt = (w // 2, h // 2)

    if point_prompt is not None:
        pts = np.array([point_prompt], dtype=np.int32)
        labels = np.array([1], dtype=np.int32)
        masks, scores, _ = predictor.predict(
            point_coords=pts,
            point_labels=labels,
            multimask_output=multimask_output
        )
    else:
        x1, y1, x2, y2 = box_prompt
        box = np.array([x1, y1, x2, y2], dtype=np.int32)
        masks, scores, _ = predictor.predict(
            box=box[None, :],
            multimask_output=multimask_output
        )

    best = np.argmax(scores)
    mask = masks[best].astype(np.uint8) * 255
    return mask

def largest_contour(binary: np.ndarray, min_area: int = MIN_AREA):
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]
    if not cnts:
        return None
    return max(cnts, key=cv2.contourArea)

def contour_to_polygon(contour: np.ndarray, epsilon_ratio: float = EPSILON_RATIO) -> np.ndarray:
    peri = cv2.arcLength(contour, True)
    epsilon = peri * epsilon_ratio
    poly = cv2.approxPolyDP(contour, epsilon, True)
    return poly.reshape(-1, 2)

def polygon_to_four_corners(poly: np.ndarray) -> np.ndarray:
    if poly.shape[0] == 4 and not USE_MIN_AREA_RECT:
        return order_corners(poly.copy())
    rect = cv2.minAreaRect(poly.astype(np.float32))
    box = cv2.boxPoints(rect)
    return order_corners(np.int32(box))

def order_corners(pts: np.ndarray) -> np.ndarray:
    pts = pts.astype(np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(d)]
    bl = pts[np.argmax(d)]
    return np.array([tl, tr, br, bl], dtype=np.int32)

def draw_polygon(img_bgr: np.ndarray, poly: np.ndarray, color=(0,255,0), thickness=3):
    vis = img_bgr.copy()
    cv2.polylines(vis, [poly.reshape(-1,1,2)], True, color, thickness)
    for i,(x,y) in enumerate(poly):
        cv2.circle(vis, (int(x),int(y)), 6, (0,0,255), -1)
        cv2.putText(vis, str(i), (int(x)+6,int(y)-6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2, cv2.LINE_AA)
    return vis

# ============== メイン処理 ==============
def process_image(
    img_path: Path,
    predictor: SamPredictor,
    point_prompt: Optional[Tuple[int,int]] = None,
    box_prompt: Optional[Tuple[int,int,int,int]] = None,
    save_debug: bool = True
):
    img = cv2.imread(str(img_path))
    assert img is not None, f"Failed to read: {img_path}"

    mask = predict_mask(predictor, img, point_prompt=point_prompt, box_prompt=box_prompt)

    cnt = largest_contour(mask)
    if cnt is None:
        print(f"[WARN] No contour found (area<{MIN_AREA}).")
        return

    poly = contour_to_polygon(cnt)
    corners4 = polygon_to_four_corners(poly)

    overlay = img.copy()
    overlay[mask>0] = (overlay[mask>0] * 0.5 + np.array([0,255,0])*0.5).astype(np.uint8)

    vis_poly = draw_polygon(img, poly, color=(0,200,255), thickness=2)
    vis_corners = draw_polygon(img, corners4, color=(0,255,0), thickness=3)

    if save_debug:
        out_dir = img_path.parent / "out_mobilesam"
        out_dir.mkdir(exist_ok=True)
        cv2.imwrite(str(out_dir / f"{img_path.stem}_mask.png"), mask)
        cv2.imwrite(str(out_dir / f"{img_path.stem}_overlay.png"), overlay)
        cv2.imwrite(str(out_dir / f"{img_path.stem}_poly.png"), vis_poly)
        cv2.imwrite(str(out_dir / f"{img_path.stem}_corners.png"), vis_corners)

    print("== Result ==")
    print(f"polygon (N={poly.shape[0]}):\n{poly}")
    print(f"corners4 (tl,tr,br,bl):\n{corners4}")

def main():
    predictor = load_mobilesam(MOBILE_SAM_WEIGHTS, model_type=MODEL_TYPE, device=DEVICE)
    process_image(IMG_PATH, predictor, point_prompt=POINT_PROMPT, box_prompt=BOX_PROMPT, save_debug=True)

if __name__ == "__main__":
    main()
